In [1]:
!pip install google-generativeai chromadb

INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
  Using cached googleapis_common_protos-1.75.0-py3-none-any.whl.metadata (8.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.82.1-py3-none-any.whl.metadata (1.2 kB)
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO

In [ ]:
import google.generativeai as genai

# Authenticate with your Gemini API key
genai.configure(api_key="YOUR_GEMINI_API_KEY")

# Generate an embedding for a text string
def get_embedding(text):
    client = genai.Client()
    result = client.models.embed_content(
        model="text-embedding-004",  # Use the latest available embedding model
        contents=text
    )
    return result.embeddings

In [ ]:
import chromadb

# Initialize ChromaDB client and collection
client = chromadb.Client()
collection = client.create_collection("my_docs")

# Example data
documents = [
    {"id": "1", "text": "Zomato is a food delivery company."},
    {"id": "2", "text": "AWS Lambda is used for serverless computing."},
    {"id": "3", "text": "GARCH models are used in financial time series analysis."}
]

# Add documents with embeddings
for doc in documents:
    embedding = get_embedding(doc["text"])
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["text"]]
    )

In [ ]:
query = "What is GARCH used for?"
query_embedding = get_embedding(query)

# Retrieve top 2 most similar documents
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

for doc, score in zip(results['documents'][0], results['distances'][0]):
    print(f"Document: {doc}\nSimilarity Score: {score}\n")

In [ ]:
# Concatenate retrieved docs as context
context = "\n".join(results['documents'][0])

# Generate answer using Gemini LLM
llm = genai.GenerativeModel("gemini-1.5-flash-latest")
response = llm.generate_content(
    f"Based on the following context, answer the question: '{query}'\nContext:\n{context}"
)
print(response.text)